<div style="padding: 20px; background: linear-gradient(90deg, #11998e 0%, #38ef7d 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">⚙️ Module 5.4: Hybrid Search (BM25 + Semantic)</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Combining exact keyword matching with semantic meaning for the ultimate retrieval engine.</p>
</div>

---

## 1. The Limitation of Pure Semantic Search

Dense Embeddings (Semantic Search) are incredible at understanding meaning. However, they **fail terribly** at:
- Exact names (e.g., "Jane Doe")
- Product IDs or Serial Numbers (e.g., "Model #XQ-9000")
- Acronyms

If a user searches for `Error Code 8042`, semantic search might return documents about `Error Code 8043` because they are semantically similar concepts (both are error codes). 

## 2. The Solution: Hybrid Search
Hybrid search combines:
1. **Dense Vectors (Semantic)**: Good at "Concepts"
2. **Sparse Vectors (BM25 Keyword)**: Good at "Exact Matches"

Instead of relying on a black-box library, let's build our own simple Hybrid Search engine to see exactly how it works under the hood!

In [1]:
# !pip install rank_bm25
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

docs = [
    Document(page_content="Model XQ-9000 is a high-performance blender."),
    Document(page_content="Model XQ-8000 is a low-performance blender."),
    Document(page_content="The NutriBullet is a blender that mixes fruit semantically well."),
]

# 1. Setup Keyword Retriever (BM25)
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 2

# 2. Setup Semantic Retriever (Chroma)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vs = Chroma.from_documents(docs, embeddings, collection_name="hybrid_demo")
semantic_retriever = vs.as_retriever(search_kwargs={"k": 2})

print("Both individual retrievers ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Both individual retrievers ready.


## 3. Building the Hybrid Function
To combine them, we run both searches, rank the results, and merge them using **Reciprocal Rank Fusion (RRF)** or simple list combination.

In [2]:
def hybrid_search(query: str, k: int = 2):
    # 1. Get results from both
    bm25_docs = bm25_retriever.invoke(query)
    semantic_docs = semantic_retriever.invoke(query)
    
    # 2. Combine and deduplicate
    combined = []
    seen = set()
    
    # Interleave results: 1st from Semantic, 1st from BM25, 2nd from Semantic, etc.
    for sem_d, bm_d in zip(semantic_docs, bm25_docs):
        if sem_d.page_content not in seen:
            combined.append(sem_d)
            seen.add(sem_d.page_content)
        if bm_d.page_content not in seen:
            combined.append(bm_d)
            seen.add(bm_d.page_content)
            
    # Return top K
    return combined[:k]

### Let's test the Edge Case

In [3]:
query = "I need help with Model XQ-9000"

print(f"Query: '{query}'\n")

# Semantic struggles with exact numbers
print("--- Pure Semantic ---")
for d in semantic_retriever.invoke(query):
    print(f"• {d.page_content}")

# BM25 nails the exact number
print("\n--- Pure Keyword (BM25) ---")
for d in bm25_retriever.invoke(query):
    print(f"• {d.page_content}")

# Our Hybrid gets the best of both worlds
print("\n--- Hybrid Search ---")
for d in hybrid_search(query):
    print(f"• {d.page_content}")

Query: 'I need help with Model XQ-9000'

--- Pure Semantic ---
• Model XQ-9000 is a high-performance blender.
• Model XQ-8000 is a low-performance blender.

--- Pure Keyword (BM25) ---
• Model XQ-9000 is a high-performance blender.
• Model XQ-8000 is a low-performance blender.

--- Hybrid Search ---
• Model XQ-9000 is a high-performance blender.
• Model XQ-8000 is a low-performance blender.
